# Prompt Engineering
In diesem Notebook werden wir das Konzept des Prompt Engineering erkunden.

## Aufbau eines Prompts
Ein Prompt ist eine strukturierte Eingabe, die die Antwort des Modells steuert. Er besteht typischerweise aus:
- **Anweisung**: Eine klare Direktive, was das Modell tun soll. Typischerweise, wie es Eingaben und/oder externe Informationen nutzen soll, um die gewünschte Ausgabe zu erzeugen.
- **Externe Informationen oder Kontext**: Zusätzliche Daten, die wir dem Modell-Prompt manuell bereitstellen, aus einer Vektordatenbank abrufen oder über eine Websuche erhalten können.
- **Benutzereingabe**: Die spezifische Abfrage, die der Nutzer eingibt.
- **Ausgabeindikator**: Der Beginn der Modellantwort, der ein bestimmter Ausdruck oder ein Format sein kann, dem das Modell folgen soll. Zum Beispiel: "Die Antwort lautet:" oder "Antwort:"

Nicht alle Prompts werden alle diese Komponenten haben, aber ein gut strukturierter Prompt enthält typischerweise mindestens zwei oder mehr davon.

In [ ]:
from dotenv import load_dotenv
import warnings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [ ]:
load_dotenv()

In [ ]:
prompt = """Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Geoffrey Everest Hinton (born 6 December 1947) is a British-Canadian computer scientist, cognitive scientist, 
cognitive psychologist, known for his work on artificial neural networks which earned him the title as the 
"Godfather of AI". Hinton is University Professor Emeritus at the University of Toronto. From 2013 to 2023, 
he divided his time working for Google (Google Brain) and the University of Toronto, before publicly announcing 
his departure from Google in May 2023, citing concerns about the risks of artificial intelligence (AI) technology.

In 2017, he co-founded and became the chief scientific advisor of the Vector Institute in Toronto.
With David Rumelhart and Ronald J. Williams, Hinton was co-author of a highly cited paper published in 1986 
that popularised the backpropagation algorithm for training multi-layer neural networks, although they were 
not the first to propose the approach. Hinton is viewed as a leading figure in the deep learning community.
The image-recognition milestone of the AlexNet designed in collaboration with his students Alex Krizhevsky 
and Ilya Sutskever for the ImageNet challenge 2012[22] was a breakthrough in the field of computer vision.

Hinton received the 2018 Turing Award, often referred to as the "Nobel Prize of Computing", together with 
Yoshua Bengio and Yann LeCun, for their work on deep learning. They are sometimes referred to as the 
"Godfathers of Deep Learning", and have continued to give public talks together. He was also awarded 
the 2024 Nobel Prize in Physics, shared with John Hopfield.

Question: Which important achievements did he get?

Answer: """

In diesem Beispiel haben wir:
* **Anweisung**: "Beantworte die Frage basierend auf dem bereitgestellten Kontext."
* **Externe Informationen oder Kontext**: Der bereitgestellte Text über Hinton.
* **Benutzereingabe**: "Welche Beiträge hat er geleistet?"
* **Ausgabeindikator**: "Answer:"

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1,)

In [ ]:
# make a response from our prompt
answer = llm.invoke(prompt)   
# print the response
print(answer.content.strip())

Normalerweise kennen wir die Benutzereingabe nicht im Voraus. Anstatt den vollständigen Prompt zu schreiben, können wir eine Vorlage verwenden, um ihn dynamisch zu erstellen.

In [ ]:
prompt = """Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Geoffrey Everest Hinton (born 6 December 1947) is a British-Canadian computer scientist, cognitive scientist, 
cognitive psychologist, known for his work on artificial neural networks which earned him the title as the 
"Godfather of AI". Hinton is University Professor Emeritus at the University of Toronto. From 2013 to 2023, 
he divided his time working for Google (Google Brain) and the University of Toronto, before publicly announcing 
his departure from Google in May 2023, citing concerns about the risks of artificial intelligence (AI) technology.

In 2017, he co-founded and became the chief scientific advisor of the Vector Institute in Toronto.
With David Rumelhart and Ronald J. Williams, Hinton was co-author of a highly cited paper published in 1986 
that popularised the backpropagation algorithm for training multi-layer neural networks, although they were 
not the first to propose the approach. Hinton is viewed as a leading figure in the deep learning community.
The image-recognition milestone of the AlexNet designed in collaboration with his students Alex Krizhevsky 
and Ilya Sutskever for the ImageNet challenge 2012[22] was a breakthrough in the field of computer vision.

Hinton received the 2018 Turing Award, often referred to as the "Nobel Prize of Computing", together with 
Yoshua Bengio and Yann LeCun, for their work on deep learning. They are sometimes referred to as the 
"Godfathers of Deep Learning", and have continued to give public talks together. He was also awarded 
the 2024 Nobel Prize in Physics, shared with John Hopfield.

Question: {query}

Answer: """
prompt_template = PromptTemplate(
    input_variables=["query"],
    template=prompt,
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()


Der output_parser wird verwendet, um die Antwort des Modells in ein bestimmtes Format zu parsen.

In [ ]:
chain = prompt_template | llm | output_parser

answer = chain.invoke({"query": "Which important achievements did he get?"})
print(answer)

## Few-Shot Prompting
Dies ist ideal für das, was wir _Few-Shot Learning_ mit unserem Prompt nennen.
Um etwas Kontext zu geben: Die primären Wissensquellen für LLMs sind:
+ __Parametrisches Wissen__: Es wurde während der Trainingsphase erlernt und ist in den Parametern des Modells gespeichert.
+ __Quellwissen__: Es wird dem Modell zur Inferenzzeit als Eingabe bereitgestellt, z.B. über den Prompt.

Das bedeutet, dass wir den Prompt verwenden können, um dem Modell zusätzlichen Kontext oder Beispiele als _Quellwissen_ bereitzustellen, die es zur Generierung einer genaueren Antwort nutzen kann.


### Few-Shot Training

In [ ]:
from IPython.display import Markdown
template = """Create a FQA in Markdown Format on the following questions and answers,
    Q1: Do you ship internationally?,
    A1: Yes, we ship worldwide. Shipping times and costs vary depending on the destination.
    Q2: How can I track my order?,
    A2: Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.
    Q3: What is your return policy?,
    A3: You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.

    """
    

prompt_template = PromptTemplate(
    template=template,
    input_variables=[] # No input variables needed for this template
)

chain = prompt_template | llm | output_parser

answer = chain.invoke({})
Markdown(answer)

Die Ausgabe ist in Ordnung, aber was, wenn wir die Ausgabe in einem bestimmten Format formatieren möchten? Wir können dem Modell einige Beispiele des gewünschten Ausgabeformats im Prompt geben. Dies wird als Few-Shot Prompting bezeichnet.

In [ ]:
template = """
Create a structured Markdown FAQ with links, headers, and bullet points based on the following questions and answers.

**Example:**

Example Input:
Q1: Do you ship internationally?
A1: Yes, we ship worldwide. Shipping times and costs vary depending on the destination.
Q2: How can I track my order?
A2: Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.
Q3: What is your return policy?
A3: You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.

Example Output:
# Frequently Asked Questions
## Shipping
- **Do you ship internationally?**  
  Yes, we ship worldwide. Shipping times and costs vary depending on the destination.
- **How can I track my order?**  
  Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.
## Returns
- **What is your return policy?**  
  You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.

Now, create a similar FAQ based on the following questions and answers:
Q1: What payment methods are accepted?,
A1: We accept Visa, MasterCard, PayPal, Apple Pay, and Klarna (Buy Now, Pay Later in select countries).,
Q2: Are your products eco-friendly?,
A2: Yes, we use sustainable materials and practices in our production process.,
Q3: Can I change or cancel my order after placing it?,
A3: We can modify or cancel orders within 1 hour of purchase. Contact our support team immediately for assistance.
"""

In [ ]:
prompt_template = PromptTemplate(
    template=template,
    input_variables=[] # No input variables needed for this template
)

chain = prompt_template | llm | output_parser

answer = chain.invoke({})
Markdown(answer)

Wir haben eine besser lesbare Ausgabe erhalten, indem wir ein Few-Shot-Beispiel zum Prompt hinzugefügt haben.
In LangChain können wir auch FewShotPromptTemplate verwenden, um einen Few-Shot-Prompt zu erstellen.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate


In [ ]:
examples = [
    {
        'input': (
            "Q1: Do you ship internationally?\n"
            "A1: Yes, we ship worldwide. Shipping times and costs vary depending on the destination.\n"
        ),
        'output': (
            "## Shipping\n"
            "- **Do you ship internationally?**  \n"
            "  Yes, we ship worldwide. Shipping times and costs vary depending on the destination.\n"
        )
    },
    {        'input': (
            "Q2: How can I track my order?\n"
            "A2: Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.\n"
        ),
        'output': (
            "## Tracking\n"
            "- **How can I track my order?**  \n"
            "  Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.\n"
        )
    },
    {
        'input': (
            "Q3: What is your return policy?\n"
            "A3: You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.\n"
        ),
        'output': (
            "## Returns\n"
            "- **What is your return policy?**  \n"
            "  You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.\n"
        )
    }
]

In [ ]:
# Create template 
example_template = """Example input:
{input}

Example Output:
# Frequently Asked Questions\n\n
{output}
"""

In [ ]:
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template=example_template
)

In [ ]:
# Instruction prefix
prefix = """Create a structured Markdown FAQ with links, headers, and bullet points based on the following questions and answers.

**Example:**"""

# User input format and instructions
suffix = """
Now, create a similar FAQ based on the following questions and answers:
{input}"""

In [ ]:
# Create the few-shot prompt template
few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input"],
    example_separator="\n"
)

In [ ]:
user_input = """Q1: What payment methods are accepted?,
A1: We accept Visa, MasterCard, PayPal, Apple Pay, and Klarna (Buy Now, Pay Later in select countries).,
Q2: Are your products eco-friendly?,
A2: Yes, we use sustainable materials and practices in our production process.,
Q3: Can I change or cancel my order after placing it?,
A3: We can modify or cancel orders within 1 hour of purchase. Contact our support team immediately for assistance.
"""

Schauen wir uns nun an, welchen Prompt dies erzeugt, wenn wir eine Benutzeranfrage eingeben...

In [ ]:
formatted_prompt = few_shot.format(input=user_input)
print(formatted_prompt)

Wir können eine Chain verwenden, um Antworten zu generieren

In [ ]:
chain = few_shot | llm | output_parser

answer = chain.invoke({"input": user_input})
Markdown(answer)

## Fazit
In diesem Notebook haben wir gelernt:
- wie wir Prompt Engineering einsetzen können, um die Antwort des Modells zu steuern
- wie man einen Prompt mit einer Anweisung, externen Informationen, Benutzereingabe und Ausgabeindikator strukturiert
- wie man Few-Shot Prompting nutzt, um Beispiele des gewünschten Ausgabeformats bereitzustellen
- wie man LangChains FewShotPromptTemplate verwendet, um einen Few-Shot-Prompt zu erstellen